# Assignment 07 - Notebook 01
## Knowledge Base và Knowledge Graph

Notebook giải thích từng bước cách biểu diễn tri thức về đái tháo đường bằng entity, relationship, attribute và triple. Chạy các cell từ trên xuống.

## 1. Khởi tạo

Xác định thư mục dự án để notebook chạy được cả từ JupyterLab và VS Code.

In [1]:
from pathlib import Path
import json, sys
ROOT = Path.cwd()
if not (ROOT / 'knowledge').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print('Project:', ROOT)

Project: C:\DATA\assignment3\assignment7


## 2. Đọc Knowledge Base

`documents.json` là kho văn bản có tiêu đề, URL nguồn và nội dung. `graph.json` là danh sách triple `subject - relation - object`.

In [2]:
documents = json.loads((ROOT/'knowledge/documents.json').read_text(encoding='utf-8'))
triples = json.loads((ROOT/'knowledge/graph.json').read_text(encoding='utf-8'))
print('Số tài liệu:', len(documents))
print('Số triple:', len(triples))
documents[:2]

Số tài liệu: 6
Số triple: 14


[{'id': 'who-overview', 'title': 'WHO - Diabetes overview', 'url': 'https://www.who.int/news-room/fact-sheets/detail/diabetes', 'text': 'Đái tháo đường là bệnh mạn tính xảy ra khi tuyến tụy không sản xuất đủ insulin hoặc cơ thể không sử dụng hiệu quả insulin. Tăng đường huyết kéo dài có thể gây tổn thương tim, mạch máu, mắt, thận và thần kinh. Type 1 liên quan thiếu insulin và cần dùng insulin; type 2 thường liên quan tình trạng cơ thể sử dụng insulin không hiệu quả.'}, {'id': 'cdc-risk', 'title': 'CDC - Type 2 diabetes risk factors', 'url': 'https://www.cdc.gov/diabetes/risk-factors/index.html', 'text': 'Các yếu tố liên quan nguy cơ đái tháo đường type 2 gồm thừa cân hoặc béo phì, ít vận động, tiền sử gia đình, tuổi cao hơn, tăng huyết áp và tiền đái tháo đường. Nguy cơ thống kê không phải chẩn đoán cá nhân.'}]

## 3. Entity - Relationship - Attribute

Entity là đối tượng như `type 2`, relationship là liên hệ như `có yếu tố nguy cơ`, còn thuộc tính mô tả entity. Triple tạo một cạnh có hướng trong đồ thị.

In [3]:
for t in triples[:8]:
    print(f"({t['subject']}) -[{t['relation']}]-> ({t['object']})")

(đái tháo đường) -[có loại]-> (type 1)
(đái tháo đường) -[có loại]-> (type 2)
(type 1) -[liên quan]-> (thiếu insulin)
(type 2) -[liên quan]-> (đề kháng insulin)
(type 2) -[có yếu tố nguy cơ]-> (thừa cân)
(type 2) -[có yếu tố nguy cơ]-> (ít vận động)
(type 2) -[có yếu tố nguy cơ]-> (tăng huyết áp)
(đái tháo đường) -[được đánh giá bằng]-> (HbA1c)


## 4. Dựng danh sách kề

Danh sách kề giúp duyệt từ một thực thể sang các thực thể liên quan. Đây là bản đơn giản của property graph; khi triển khai lớn có thể chuyển sang Neo4j và Cypher.

In [4]:
from collections import defaultdict
adj = defaultdict(list)
for t in triples:
    adj[t['subject'].lower()].append((t['relation'], t['object']))
adj['đái tháo đường']

[('có loại', 'type 1'), ('có loại', 'type 2'), ('được đánh giá bằng', 'HbA1c'), ('được đánh giá bằng', 'glucose huyết tương'), ('có thể gây', 'bệnh tim mạch'), ('có thể gây', 'bệnh thận'), ('có thể gây', 'tổn thương thần kinh')]

## 5. Truy vấn multi-hop

Câu hỏi nhiều bước cần đi qua hơn một cạnh, ví dụ từ `type 2` tới yếu tố nguy cơ rồi tới hành động cải thiện.

In [5]:
from src.rag_agent import KnowledgeGraph
kg = KnowledgeGraph(triples)
paths = kg.expand('type 2 thừa cân ít vận động', max_hops=2)
paths

[{'subject': 'type 1', 'relation': 'liên quan', 'object': 'thiếu insulin', 'hop': 1}, {'subject': 'type 2', 'relation': 'liên quan', 'object': 'đề kháng insulin', 'hop': 1}, {'subject': 'type 2', 'relation': 'có yếu tố nguy cơ', 'object': 'thừa cân', 'hop': 1}, {'subject': 'type 2', 'relation': 'có yếu tố nguy cơ', 'object': 'ít vận động', 'hop': 1}, {'subject': 'type 2', 'relation': 'có yếu tố nguy cơ', 'object': 'tăng huyết áp', 'hop': 1}, {'subject': 'thừa cân', 'relation': 'có thể cải thiện bằng', 'object': 'dinh dưỡng hợp lý', 'hop': 1}, {'subject': 'ít vận động', 'relation': 'có thể cải thiện bằng', 'object': 'vận động thể lực', 'hop': 1}]

## 6. Nhận xét

Knowledge Graph biểu diễn quan hệ rõ và hỗ trợ suy luận nhiều bước, nhưng tốn công xây dựng schema và kiểm soát chất lượng. Trong bài này graph nhỏ được lưu JSON để notebook chạy độc lập; Neo4j là hướng mở rộng phù hợp khi số node và truy vấn tăng.